# 6. Ensemble Modeling

**Purpose**: Combine the four tree models from Part 5 (CatBoost, XGBoost, LightGBM, RandomForest) with a freshly retrained Logistic Regression into a weighted ensemble. This ensemble improves predictive performance by leveraging the strengths of diverse models, especially in terms of PR‑AUC and log loss.

**Workflow**:
1. Load pre‑trained tree models from Part 5 (`models_train_only/`).
2. Retrain Logistic Regression from scratch using the exact Part 5 pipeline (median imputation + StandardScaler).
3. Use the time series CV performance from part 5 to derive ensemble weights.
4. Select the best ensemble configuration (all models vs. tree‑only) based on validation PR‑AUC.
5. Refit all winning models on the combined `train+val` set using Part 5 parameters.
6. Evaluate the final ensemble once on the held‑out test set.
7. Save predictions.

This approach ensures consistency with Part 5 while avoiding serialization issues with the logistic regression pipeline.

## 1. Setup

### 1a. Install CatBoost (if missing)

In [ ]:
try:
    import catboost
except ImportError:
    !pip install -q catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.9 MB/s eta 0:00:00


### 1b. Imports and Global Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, gc, joblib
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 100)
print("All libraries imported.")

# === CONFIGURATION ===
# Base data directory
DATA_DIR = "../data/"

# Processed data location (train/val/test parquet files)
PROCESSED_DIR = os.path.join(DATA_DIR, "processed/")

# Results saved directly into results/
RESULTS_DIR = os.path.join(DATA_DIR, "results/")

# Pre‑trained tree models from Part 5 (stored in results/)
MODELS_DIR = os.path.join(RESULTS_DIR, "models_train_only/")



# Features and target (must match Part 5)
FEATURES = [
    'avg_plan_price',
    'avg_payment_per_day',
    'days_since_first_trans',
    'days_since_last_use',
    'ratio_auto_renew',
    'total_secs_velocity',
    'num_unq_velocity',
    'last_is_auto_renew',
    'last_is_cancel',
]
TARGET = 'is_churn'
RANDOM_STATE = 42

# Subsampling for quick tests – set to None for full data
SAMPLE_FRAC = 0.15   # e.g., 0.15 for 15% sample

print(f"Using features: {FEATURES}")
print(f"SAMPLE_FRAC = {SAMPLE_FRAC if SAMPLE_FRAC else 'None (full data)'}")
print(f"Data dir: {DATA_DIR}")
print(f"Models dir: {MODELS_DIR}")
print(f"Results dir: {RESULTS_DIR}")


All libraries imported.
Using features: ['avg_plan_price', 'avg_payment_per_day', 'days_since_first_trans', 'days_since_last_use', 'ratio_auto_renew', 'total_secs_velocity', 'num_unq_velocity', 'last_is_auto_renew', 'last_is_cancel']
SAMPLE_FRAC = 0.15
Data dir: /content/drive/MyDrive/Colab Notebooks/Churn Prediction/Data/
Models dir: /content/drive/MyDrive/Colab Notebooks/Churn Prediction/Data/new_data_model_results/models_train_only
Results dir: /content/drive/MyDrive/Colab Notebooks/Churn Prediction/Data/model_results


## 2. Load Data
Read the processed train/val/test parquet files from `PROCESSED_DIR`. If `SAMPLE_FRAC` is set, we subsample the training set only (validation and test remain full).

We also downcast numeric columns to save memory (same as Part 5).

In [ ]:
def shrink(df):
    for c in df.select_dtypes(include=['float64']).columns:
        df[c] = df[c].astype('float32')
    for c in df.select_dtypes(include=['int64']).columns:
        df[c] = pd.to_numeric(df[c], downcast='integer')
    return df

def load_data(file_name, frac=None, random_state=42):
    path = os.path.join(PROCESSED_DIR, file_name)
    if frac is None or frac >= 1.0:
        df = pd.read_parquet(path)
        print(f"  Loaded full {file_name}: {df.shape[0]:,} rows")
    else:
        # Sample row groups for speed
        import pyarrow.parquet as pq
        pf = pq.ParquetFile(path)
        n_groups = pf.num_row_groups
        n_to_read = max(1, int(n_groups * frac))
        indices = sorted(np.random.choice(n_groups, size=n_to_read, replace=False))
        table = pf.read_row_groups(indices)
        df = table.to_pandas()
        print(f"  Loaded {df.shape[0]:,} rows from {file_name} ({frac*100:.0f}%)")
    return shrink(df)

print("Loading data from:", PROCESSED_DIR)
df_train = load_data("train.parquet", frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
df_val   = load_data("val.parquet")
df_test  = load_data("test.parquet")

print(f"Train: {df_train.shape} | Churn rate: {df_train[TARGET].mean():.4f}")
print(f"Val:   {df_val.shape}   | Churn rate: {df_val[TARGET].mean():.4f}")
print(f"Test:  {df_test.shape}  | Churn rate: {df_test[TARGET].mean():.4f}")

# Keep copies for later refitting on train+val
df_train_orig = df_train.copy()
df_val_orig = df_val.copy()

X_train = df_train[FEATURES]; y_train = df_train[TARGET]
X_val   = df_val[FEATURES]; y_val = df_val[TARGET]
X_test  = df_test[FEATURES]; y_test = df_test[TARGET]
test_msno = df_test['msno'].copy()

del df_train, df_val, df_test
_ = gc.collect()

Loading data from: /content/drive/MyDrive/Colab Notebooks/Churn Prediction/Data/processed
  Loaded 1,048,576 rows from train.parquet (15%)
  Loaded full val.parquet: 1,708,636 rows
  Loaded full test.parquet: 2,606,258 rows
Train: (1048576, 15) | Churn rate: 0.1192
Val:   (1708636, 15)   | Churn rate: 0.0652
Test:  (2606258, 15)  | Churn rate: 0.0551


## 3. Load Pre‑trained Tree Models from Part 5
Load the four tree models (CatBoost, XGBoost, LightGBM, RandomForest) that were saved in `models_train_only/` during Part 5.

In [ ]:
print("Loading pre‑trained tree models from Part 5...")
model_files = {
    'CatBoost': 'CatBoost.pkl',
    'XGBoost': 'XGBoost.pkl',
    'LightGBM': 'LightGBM.pkl',
    'RandomForest': 'RandomForest.pkl',
}
loaded_models = {}
for name, fname in model_files.items():
    path = os.path.join(MODELS_DIR, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Model file not found: {path}")
    loaded_models[name] = joblib.load(path)
    print(f"  Loaded {name}")

Loading pre‑trained tree models from Part 5...
  Loaded CatBoost
  Loaded XGBoost
  Loaded LightGBM
  Loaded RandomForest


## 4. Retrain Logistic Regression from Scratch
To avoid serialisation issues, we retrain Logistic Regression using the exact same pipeline as Part 5: median imputation + StandardScaler + logistic regression with `max_iter=1000`.

In [ ]:
print("Retraining Logistic Regression with Part 5 parameters...")
lr_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
lr_pipeline.fit(X_train, y_train)
print("  LogisticRegression retrained successfully.")
loaded_models['LogisticRegression'] = lr_pipeline

Retraining Logistic Regression with Part 5 parameters...
  LogisticRegression retrained successfully.


## 5. Compute Predictions and Individual Scores
Get validation predictions from all five models, then print individual performance metrics for reference.

In [ ]:
def get_predictions(models, X):
    preds = {}
    for name, model in models.items():
        preds[name] = model.predict_proba(X)[:, 1]
    return preds

preds_val = get_predictions(loaded_models, X_val)

print("\nIndividual model performance (using pre‑trained tree models + retrained LR):")
scores = {}
for name, p in preds_val.items():
    roc = roc_auc_score(y_val, p)
    pr = average_precision_score(y_val, p)
    ll = log_loss(y_val, p)
    scores[name] = {'val_roc': roc, 'val_pr': pr, 'val_ll': ll}
    print(f"  {name}: VAL ROC={roc:.4f}, PR={pr:.4f}, LL={ll:.4f}")


Individual model performance (using pre‑trained tree models + retrained LR):
  CatBoost: VAL ROC=0.9379, PR=0.7580, LL=0.1089
  XGBoost: VAL ROC=0.9402, PR=0.7588, LL=0.1084
  LightGBM: VAL ROC=0.9401, PR=0.7563, LL=0.1090
  RandomForest: VAL ROC=0.9377, PR=0.7576, LL=0.1103
  LogisticRegression: VAL ROC=0.8990, PR=0.6941, LL=0.1390


## 6. Build Ensemble Weights from time series CV PR‑AUC
We compute weights for two configurations:
- **ALL**: all 5 models (including LR)
- **TREE‑ONLY**: only the 4 tree models

The weights are derived from time-series CV PR‑AUC performance (higher PR‑AUC → higher weight).

In [ ]:
# ---- Weight builder from CV results ----
def build_weights_from_cv(csv_path, model_name_map):
    """
    Reads cv_comparison.csv and returns weights (softmax‑scaled PR‑AUC).
    model_name_map: dict mapping CSV index names -> notebook model keys.
    """
    df_cv = pd.read_csv(csv_path, index_col=0)   # index is model name
    # Extract PR‑AUC column
    pr_auc = df_cv['pr_auc']
    # Map names
    mapped = {}
    for csv_name, notebook_name in model_name_map.items():
        if csv_name in pr_auc.index:
            mapped[notebook_name] = pr_auc[csv_name]
        else:
            print(f"Warning: {csv_name} not found in CV results.")
    # Softmax scaling (same as before)
    values = np.array(list(mapped.values()))
    scaled = (values - values.min()) / (values.max() - values.min() + 1e-8)
    exp_vals = np.exp(scaled / 0.5)
    weights = {name: exp_vals[i] / exp_vals.sum() for i, name in enumerate(mapped.keys())}
    return weights

model_name_map = {
    'CatBoost': 'CatBoost',
    'XGBoost': 'XGBoost',
    'LightGBM': 'LightGBM',
    'RandomForest': 'RandomForest',
    'LogReg': 'LogisticRegression',
}

# ---- Import the performance results from notebook 5 ----
cv_path = os.path.join(RESULTS_DIR, "cv_comparison.csv")
if not os.path.exists(cv_path):
    raise FileNotFoundError(f"CV results not found: {cv_path}")

# Compute weights for all models (including LR)
weights_all = build_weights_from_cv(cv_path, model_name_map)

# Tree‑only weights (exclude LogisticRegression)
weights_tree = {k: v for k, v in weights_all.items() if k != 'LogisticRegression'}
# Renormalize tree weights to sum to 1
sum_tree = sum(weights_tree.values())
weights_tree = {k: v/sum_tree for k, v in weights_tree.items()}

print("\nWeights derived from CV PR‑AUC:")
print("ALL models:")
for k, v in weights_all.items():
    print(f"  {k}: {v:.4f}")
print("TREE‑ONLY:")
for k, v in weights_tree.items():
    print(f"  {k}: {v:.4f}")

# Function to apply ensemble weights to a dictionary of predictions
def apply_ensemble(predictions, weights):
    ens = np.zeros_like(predictions[list(predictions.keys())[0]])
    for name, preds in predictions.items():
        if name in weights:
            ens += weights[name] * preds
    return ens

# Evaluate both ensembles on the validation set
val_all = apply_ensemble(preds_val, weights_all)
val_tree = apply_ensemble(preds_val, weights_tree)

val_all_pr = average_precision_score(y_val, val_all)
val_tree_pr = average_precision_score(y_val, val_tree)

print(f"\nValidation ensemble performance (PR‑weighted):")
print(f"  ALL (incl. LR): PR={val_all_pr:.4f}")
print(f"  TREE-ONLY:      PR={val_tree_pr:.4f}")

# Choose the better configuration based on validation PR‑AUC
if val_tree_pr > val_all_pr:
    winner = "TREE-ONLY"
    winner_weights = weights_tree
else:
    winner = "ALL MODELS (incl. LR)"
    winner_weights = weights_all

print(f"\nWINNER (by PR‑AUC on VAL): {winner}")
print("Weights:")
for name, w in winner_weights.items():
    print(f"  {name}: {w:.4f}")


Weights derived from CV PR‑AUC:
ALL models:
  CatBoost: 0.2510
  XGBoost: 0.2089
  LightGBM: 0.2013
  RandomForest: 0.2984
  LogisticRegression: 0.0404
TREE‑ONLY:
  CatBoost: 0.2616
  XGBoost: 0.2177
  LightGBM: 0.2097
  RandomForest: 0.3110

Validation ensemble performance (PR‑weighted):
  ALL (incl. LR): PR=0.7605
  TREE-ONLY:      PR=0.7602

WINNER (by PR‑AUC on VAL): ALL MODELS (incl. LR)
Weights:
  CatBoost: 0.2510
  XGBoost: 0.2089
  LightGBM: 0.2013
  RandomForest: 0.2984
  LogisticRegression: 0.0404


## 7. Refit Winning Models on TRAIN+VAL
To maximise data usage, we refit each winning model on the combined `train+val` set. We use the exact Part 5 model parameters. CatBoost is refitted with a validation split for early stopping (as in Part 5).

In [ ]:
print("\nRefitting winning models on TRAIN+VAL with Part 5 parameters...")
df_full_train = pd.concat([df_train_orig, df_val_orig], ignore_index=True)
X_full = df_full_train[FEATURES]
y_full = df_full_train[TARGET]
print(f"Full training set size: {X_full.shape[0]:,}")

# Define model builders (identical to Part 5) – CatBoost already has verbose=False
def make_logreg():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ])

def make_rf():
    return RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=50,
        random_state=RANDOM_STATE, n_jobs=-1,
    )

def make_xgb():
    return xgb.XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', tree_method='hist',
        random_state=RANDOM_STATE, n_jobs=-1,
    )

def make_lgbm():
    return lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.05, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1,
    )

def make_catboost():
    return CatBoostClassifier(
        iterations=500, learning_rate=0.05, depth=6,
        random_state=RANDOM_STATE, verbose=False,   # <-- silent
    )

all_builders = {
    'CatBoost': make_catboost,
    'XGBoost': make_xgb,
    'LightGBM': make_lgbm,
    'RandomForest': make_rf,
    'LogisticRegression': make_logreg,
}

winning_model_names = list(winner_weights.keys())
refit_models = {}
for name in winning_model_names:
    print(f"  Refitting {name} on TRAIN+VAL...")
    model = all_builders[name]()
    model.fit(X_full, y_full)
    refit_models[name] = model
    gc.collect()


Refitting winning models on TRAIN+VAL with Part 5 parameters...
Full training set size: 2,757,212
  Refitting CatBoost on TRAIN+VAL...
  Refitting XGBoost on TRAIN+VAL...
  Refitting LightGBM on TRAIN+VAL...
  Refitting RandomForest on TRAIN+VAL...
  Refitting LogisticRegression on TRAIN+VAL...


## 8. Final Evaluation on Test Set
Evaluate the refitted ensemble on the untouched test set. We also print individual refitted model scores for comparison.

In [ ]:
print("\nFINAL EVALUATION ON TEST SET")
final_ensemble_test = np.zeros(len(y_test))
for name, model in refit_models.items():
    preds = model.predict_proba(X_test)[:, 1]
    final_ensemble_test += winner_weights[name] * preds

final_roc = roc_auc_score(y_test, final_ensemble_test)
final_pr = average_precision_score(y_test, final_ensemble_test)
final_ll = log_loss(y_test, final_ensemble_test)

print(f"\nFINAL MODEL ({winner} refitted on TRAIN+VAL):")
print(f"  ROC-AUC: {final_roc:.4f}")
print(f"  PR-AUC:  {final_pr:.4f}")
print(f"  Log Loss: {final_ll:.4f}")

print("\nIndividual refitted models on TEST:")
for name, model in refit_models.items():
    preds = model.predict_proba(X_test)[:, 1]
    roc = roc_auc_score(y_test, preds)
    pr = average_precision_score(y_test, preds)
    ll = log_loss(y_test, preds)
    print(f"  {name}: ROC={roc:.4f}, PR={pr:.4f}, LL={ll:.4f}")


FINAL EVALUATION ON TEST SET

FINAL MODEL (ALL MODELS (incl. LR) refitted on TRAIN+VAL):
  ROC-AUC: 0.8376
  PR-AUC:  0.5445
  Log Loss: 0.1429

Individual refitted models on TEST:
  CatBoost: ROC=0.8380, PR=0.5415, LL=0.1453
  XGBoost: ROC=0.8363, PR=0.5385, LL=0.1477
  LightGBM: ROC=0.8355, PR=0.5400, LL=0.1474
  RandomForest: ROC=0.8340, PR=0.5421, LL=0.1437
  LogisticRegression: ROC=0.8400, PR=0.4936, LL=0.1523


## 9. Save Final Predictions
Save the ensemble predictions for each customer (`msno`) along with individual model predictions (for the winning models) to a CSV file.

In [ ]:
prediction_df = pd.DataFrame({
    'msno': test_msno,
    'churn_probability': final_ensemble_test,
    'ensemble_type': winner,
})
for name, preds in preds_test.items():
    if name in winning_model_names:
        prediction_df[f"pred_{name}"] = preds

pred_path = os.path.join(RESULTS_DIR, "ensemble_final_predictions_winner.csv")
prediction_df.to_csv(pred_path, index=False)
print(f"Final predictions saved to: {pred_path}")

NameError: name 'preds_test' is not defined

## 10. Summary
Print a final summary of the ensemble run.

In [ ]:
# ---- Cell 10: Summary (simplified, no val metrics) ----
print("\n" + "="*70)
print("ENSEMBLE NOTEBOOK COMPLETE")
print("="*70)
print(f"\nData used: {'FULL' if SAMPLE_FRAC is None else f'{SAMPLE_FRAC*100:.0f}% subsample'}")
print(f"Final training rows: {X_full.shape[0]:,}")
print(f"Test rows: {X_test.shape[0]:,}")
print(f"\nWinning ensemble: {winner}")
print("Weights:")
for name, w in winner_weights.items():
    print(f"  {name}: {w:.4f}")
print(f"\nFinal Performance on TEST:")
print(f"  ROC-AUC: {final_roc:.4f}")
print(f"  PR-AUC:  {final_pr:.4f}")
print(f"  Log Loss: {final_ll:.4f}")
print(f"\nResults saved to: {RESULTS_DIR}")
print("  - final_predictions_winner.csv (predictions for all test customers)")
print("  - ensemble_meta.json (weights and metrics)")
print("="*70)

# Save ensemble metadata (test metrics only)
import json
ensemble_meta = {
    'winner': winner,
    'weights': winner_weights,
    'ensemble_test_metrics': {'roc': final_roc, 'pr': final_pr, 'll': final_ll},
}
with open(os.path.join(RESULTS_DIR, 'ensemble_meta.json'), 'w') as f:
    json.dump(ensemble_meta, f, indent=2)
print(f"  - ensemble_meta.json saved.")